# ELN: Adelie Penguin Biometrics Analysis

**Experiment ID:** EXP-2026-PENGUIN-001-Adelie  
**Operator:** Dr. Maria Okonkwo  
**Date:** 2026-02-06  
**Institution:** University of Cape Town, Marine Biology Lab

This notebook analyzes biometric measurements of Adelie penguins including culmen length, culmen depth, flipper length, and body mass, stratified by sex and island location.

In [1]:
import json
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================================
# EXPERIMENT METADATA (ELN Header)
# ============================================================================

experiment_metadata = {
    'experiment_id': 'EXP-2026-PENGUIN-001-Adelie',
    'project': 'Penguin-Biometrics-Analysis',
    'species': 'Adelie',
    'notebook_title': 'Biometric Analysis: Adelie Penguin Population',
    'date_started': '2026-02-06',
    'date_completed': '2026-02-06',
    'operator': 'Dr. Maria Okonkwo',
    'supervisor': 'Prof. David Chen',
    'institution': 'University of Cape Town, Marine Biology Lab',
    'objective': 'Analyze biometric measurements (culmen, flipper, mass) of Adelie penguins across islands and sexes',
    'hypothesis': 'Biometric measures vary significantly by sex and island location within the species',
    'data_source': 'Palmer Penguins dataset',
    'data_source_file': 'penguins_Adelie.csv',
    'data_clean_file': '',
}

experiment_metadata['data_clean_file'] = experiment_metadata['data_source_file'][0:-4] + '_clean.csv'

print("Experiment metadata loaded:")
print(f"  Experiment ID: {experiment_metadata['experiment_id']}")
print(f"  Species: {experiment_metadata['species']}")
print(f"  Operator: {experiment_metadata['operator']}")
print(f"  Data Source: {experiment_metadata['data_source_file']}")

Experiment metadata loaded:
  Experiment ID: EXP-2026-PENGUIN-001-Adelie
  Species: Adelie
  Operator: Dr. Maria Okonkwo
  Data Source: penguins_Adelie.csv


## Data Loading

Load raw biometric measurements from the Adelie-specific CSV file.

In [2]:
# Load raw data from CSV
df_raw = pd.read_csv("penguins_Adelie.csv")

print(f"Loaded {len(df_raw)} specimens of Adelie penguins")
print(f"\nData preview:")
print(df_raw.head(10))

Loaded 30 specimens of Adelie penguins

Data preview:
  species     island  culmen_length_mm  culmen_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen              39.1             18.7              181.0   
1  Adelie  Torgersen              39.5             17.4              186.0   
2  Adelie  Torgersen              40.3             18.0              195.0   
3  Adelie  Torgersen              36.7             19.3              193.0   
4  Adelie  Torgersen              39.3             20.6              190.0   
5  Adelie  Torgersen              38.9             17.8              181.0   
6  Adelie  Torgersen              39.2             19.6              195.0   
7  Adelie  Torgersen             234.1             18.1              193.0   
8  Adelie  Torgersen              42.0             -2.2              190.0   
9  Adelie  Torgersen              37.8             17.1              186.0   

   body_mass_g     sex  
0       3750.0    Male  
1       3800.0  Female  
2       3250

## Data Quality Check

In [3]:
print(f"Columns: {list(df_raw.columns)}")
print(f"\nData shape: {df_raw.shape}")


Columns: ['species', 'island', 'culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']

Data shape: (30, 7)


In [4]:
print(f"Missing values:\n{df_raw.isnull().sum()}")

Missing values:
species              0
island               0
culmen_length_mm     0
culmen_depth_mm      0
flipper_length_mm    0
body_mass_g          0
sex                  3
dtype: int64


### Data Vocabulary Check

In [5]:
from data_quality import validate_with_vocabulary, clean_errors_and_warnings

In [6]:
df, report = validate_with_vocabulary(data_path = "penguins_Adelie.csv", vocab_path= "penguin_vocabulary.csv")

In [7]:

print("Row count:", report["row_count"])
print("Errors:", report["errors"])
print("Warnings:", report["warnings"])


Row count: 30
Errors: ["Invalid categorical values in sex: ['Unknown']"]
Warnings: ['Values above max in culmen_length_mm (max=65.25): sample [234.1]', 'Values below min in culmen_depth_mm (min=10.787): sample [-2.2]']


In [8]:
clean_df, info = clean_errors_and_warnings(df, report, method="remove")
info


{'rows_removed': 3,
 'rows_remaining': 27,
 'columns_cleaned': ['species',
  'island',
  'sex',
  'culmen_length_mm',
  'culmen_depth_mm',
  'flipper_length_mm',
  'body_mass_g'],
 'mode': 'remove'}

In [9]:
print(f"After cleaning there is {len(clean_df)} specimens of Adelie penguins")

After cleaning there is 27 specimens of Adelie penguins


In [10]:
clean_df

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
4,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male
5,Adelie,Torgersen,38.9,17.8,181.0,3625.0,Female
6,Adelie,Torgersen,39.2,19.6,195.0,4675.0,Male
10,Adelie,Biscoe,37.8,18.3,174.0,3400.0,Female
11,Adelie,Biscoe,37.7,18.7,180.0,3600.0,Male
12,Adelie,Biscoe,35.9,19.2,189.0,3800.0,Female


In [11]:
clean_df.to_csv(experiment_metadata['data_clean_file'])

## Analysis: Descriptive Statistics

In [12]:
# Overall statistics
print("=" * 70)
print("OVERALL STATISTICS - ADELIE PENGUINS")
print("=" * 70)
print(clean_df[['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']].describe())

# Statistics by sex
print("\n" + "=" * 70)
print("STATISTICS BY SEX")
print("=" * 70)
print(clean_df.groupby('sex')[['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']].describe())

# Statistics by island
print("\n" + "=" * 70)
print("STATISTICS BY ISLAND")
print("=" * 70)
print(clean_df.groupby('island')[['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']].describe())

OVERALL STATISTICS - ADELIE PENGUINS
       culmen_length_mm  culmen_depth_mm  flipper_length_mm  body_mass_g
count         27.000000        27.000000          27.000000    27.000000
mean          38.911111        18.740741         185.407407  3705.555556
std            2.313395         1.224198           6.868318   394.269854
min           34.400000        16.700000         172.000000  3150.000000
25%           37.750000        17.950000         180.000000  3362.500000
50%           39.100000        18.600000         185.000000  3750.000000
75%           39.900000        19.250000         190.000000  3912.500000
max           46.000000        21.500000         197.000000  4675.000000

STATISTICS BY SEX
       culmen_length_mm                                                       \
                  count       mean       std   min   25%   50%     75%   max   
sex                                                                            
Female             13.0  37.892308  1.998108  3

In [13]:
# Calculate analysis results for export
analysis_results = {
    'species': experiment_metadata['species'],
    'specimen_count': len(clean_df),
    'specimen_by_sex': clean_df['sex'].value_counts().to_dict(),
    'specimen_by_island': clean_df['island'].value_counts().to_dict(),
    'mean_body_mass_g': float(clean_df['body_mass_g'].mean()),
    'std_body_mass_g': float(clean_df['body_mass_g'].std()),
    'mean_flipper_length_mm': float(clean_df['flipper_length_mm'].mean()),
    'mean_culmen_length_mm': float(clean_df['culmen_length_mm'].mean()),
    'mean_culmen_depth_mm': float(clean_df['culmen_depth_mm'].mean()),
    'experiment_id': experiment_metadata['experiment_id'],
    'operator': experiment_metadata['operator'],
    'date_completed': experiment_metadata['date_completed'],
    'data_source': experiment_metadata['data_source_file'],
    'data_clean': experiment_metadata['data_clean_file'],
}

# Sexual dimorphism
try:
    male_mass = clean_df[clean_df['sex'] == 'Male']['body_mass_g'].mean()
    female_mass = clean_df[clean_df['sex'] == 'Female']['body_mass_g'].mean()
    analysis_results['sexual_dimorphism'] = {
        'male_mean_body_mass_g': float(male_mass),
        'female_mean_body_mass_g': float(female_mass),
        'dimorphism_ratio': float(male_mass / female_mass),
    }
except:
    analysis_results['sexual_dimorphism'] = None

print("Analysis Results (JSON export format):")
print(json.dumps(analysis_results, indent=2))

Analysis Results (JSON export format):
{
  "species": "Adelie",
  "specimen_count": 27,
  "specimen_by_sex": {
    "Male": 14,
    "Female": 13
  },
  "specimen_by_island": {
    "Biscoe": 10,
    "Dream": 10,
    "Torgersen": 7
  },
  "mean_body_mass_g": 3705.5555555555557,
  "std_body_mass_g": 394.2698542226098,
  "mean_flipper_length_mm": 185.40740740740742,
  "mean_culmen_length_mm": 38.911111111111104,
  "mean_culmen_depth_mm": 18.740740740740737,
  "experiment_id": "EXP-2026-PENGUIN-001-Adelie",
  "operator": "Dr. Maria Okonkwo",
  "date_completed": "2026-02-06",
  "data_source": "penguins_Adelie.csv",
  "data_clean": "penguins_Adelie_clean.csv",
  "sexual_dimorphism": {
    "male_mean_body_mass_g": 3964.285714285714,
    "female_mean_body_mass_g": 3426.923076923077,
    "dimorphism_ratio": 1.1568061568061567
  }
}


## Results and Conclusions

**Key Findings for Adelie Penguins:**

- Total specimens analyzed: [from data]
- Distribution by sex: [male/female/unknown counts]
- Distribution by island: [Torgersen/Biscoe/Dream]
- Clear sexual dimorphism with males larger than females
- Geographic variation noted between islands

**Quality Assurance:**
- ✓ Data loaded and validated
- ✓ No critical missing values
- ✓ Statistical calculations complete

**Sign-Off:**
- Analyst: Dr. Maria Okonkwo
- Date: 2026-02-06
- Status: Complete - ready for aggregation

In [14]:
# Create export package for aggregation
export_package = {
    'metadata': experiment_metadata,
    'analysis_results': analysis_results,
    'specimen_count': len(clean_df),
    'export_timestamp': datetime.now().isoformat(),
    'notebook_version': '1.0'
}

print("Export package ready for ELN_results.ipynb")
print(f"Species: {analysis_results['species']}")
print(f"Specimens: {analysis_results['specimen_count']}")
print(f"Timestamp: {export_package['export_timestamp']}")

Export package ready for ELN_results.ipynb
Species: Adelie
Specimens: 27
Timestamp: 2026-03-02T11:06:54.760294
